# False Discovery Rate (FDR) - Starter Notebook
Implements the Benjamini-Hochberg procedure to control FDR in multiple hypothesis testing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
np.random.seed(0)
m = 30
n_per_group = 40
alpha = 0.05

# First 8 tests have true effects
p_values = []
for i in range(m):
    effect = 0.7 if i < 8 else 0.0
    g1 = np.random.normal(0, 1, n_per_group)
    g2 = np.random.normal(effect, 1, n_per_group)
    _, p = stats.ttest_ind(g1, g2)
    p_values.append(p)

p_values = np.array(p_values)
print('Uncorrected rejections:', (p_values < alpha).sum())

In [ ]:
def benjamini_hochberg(p_values, alpha=0.05):
    m = len(p_values)
    order = np.argsort(p_values)
    sorted_p = p_values[order]
    thresholds = alpha * np.arange(1, m + 1) / m
    # Find largest k such that p_(k) <= k*alpha/m
    rejected = sorted_p <= thresholds
    if rejected.any():
        max_k = np.max(np.where(rejected)[0])
        reject_mask = np.zeros(m, dtype=bool)
        reject_mask[order[:max_k+1]] = True
    else:
        reject_mask = np.zeros(m, dtype=bool)
    return reject_mask, thresholds, sorted_p

rejected_bh, thresholds, sorted_p = benjamini_hochberg(p_values, alpha)
print(f'BH rejections at FDR={alpha}: {rejected_bh.sum()}')
print(f'True positives (tests 1-8 rejected): {rejected_bh[:8].sum()}')
print(f'False positives (tests 9-30 rejected): {rejected_bh[8:].sum()}')

In [ ]:
# Comparison table
bonf_rejected = p_values < (alpha / m)
uncorr_rejected = p_values < alpha

comparison = pd.DataFrame({
    'Method': ['Uncorrected', 'Bonferroni (FWER)', 'Benjamini-Hochberg (FDR)'],
    'Rejections': [uncorr_rejected.sum(), bonf_rejected.sum(), rejected_bh.sum()],
    'TP (tests 1-8)': [uncorr_rejected[:8].sum(), bonf_rejected[:8].sum(), rejected_bh[:8].sum()],
    'FP (tests 9-30)': [uncorr_rejected[8:].sum(), bonf_rejected[8:].sum(), rejected_bh[8:].sum()]
})
print(comparison.to_string(index=False))

In [ ]:
# BH procedure plot
ranks = np.arange(1, m + 1)

plt.figure(figsize=(8, 5))
plt.scatter(ranks, sorted_p, c='steelblue', zorder=3, label='Sorted p-values')
plt.plot(ranks, thresholds, color='red', linestyle='--', label=f'BH threshold (k·α/m)')
plt.axhline(alpha, color='gray', linestyle=':', label=f'α={alpha} (uncorrected)')
# Shade rejected region
max_k = np.max(np.where(sorted_p <= thresholds)[0]) if (sorted_p <= thresholds).any() else -1
if max_k >= 0:
    plt.axvspan(0.5, max_k + 1.5, alpha=0.1, color='green', label='BH rejected')
plt.xlabel('Rank')
plt.ylabel('p-value')
plt.title('Benjamini-Hochberg FDR Procedure')
plt.legend()
plt.tight_layout()
plt.show()